In [1]:
import os
os.chdir('/home/smallyan/eval_agent')

# Load environment variables from .bashrc
import subprocess
result = subprocess.run(['bash', '-c', 'source /home/smallyan/.bashrc && env'], capture_output=True, text=True)
for line in result.stdout.split('\n'):
    if '=' in line:
        key, _, value = line.partition('=')
        os.environ[key] = value

# Set model cache directory
os.environ['HF_HOME'] = '/net/projects2/chai-lab/shared_models'
os.environ['TRANSFORMERS_CACHE'] = '/net/projects2/chai-lab/shared_models'

# Check GPU
import torch
print(f"Working directory: {os.getcwd()}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Working directory: /home/smallyan/eval_agent
CUDA available: True
GPU: NVIDIA A100 80GB PCIe


# Code Evaluation for InterpDetect Circuit Analysis

**Repository:** `/net/scratch2/smallyan/InterpDetect_eval`

## Overview

This notebook evaluates all code blocks in the InterpDetect implementation based on the plan.md and CodeWalkthrough.md files.

### Evaluation Criteria
- **Runnable (Y/N)**: Block executes without error
- **Correct-Implementation (Y/N)**: Logic implements described computation correctly
- **Redundant (Y/N)**: Block duplicates another block's computation
- **Irrelevant (Y/N)**: Block doesn't contribute to project goal

In [2]:
# Initialize evaluation tracking
import json
import pandas as pd

evaluation_results = []
corrections_made = 0
blocks_that_failed = 0

def record_eval(script_name, block_id, block_desc, runnable, correct, redundant, irrelevant, note=""):
    """Record evaluation result for a block"""
    global blocks_that_failed
    if runnable == "N":
        blocks_that_failed += 1
    evaluation_results.append({
        'script': script_name,
        'block_id': block_id,
        'description': block_desc,
        'runnable': runnable,
        'correct': correct,
        'redundant': redundant,
        'irrelevant': irrelevant,
        'note': note
    })
    status = "PASS" if runnable == "Y" else "FAIL"
    print(f"[{status}] {script_name}:{block_id} - {block_desc}")
    if note:
        print(f"       Note: {note}")

print("Evaluation tracking initialized")

Evaluation tracking initialized


## Part 2: Core Analysis Scripts (Training & Prediction)

These are the main scripts for the interpretability-based hallucination detection method.

In [3]:
# ===============================================
# COMPUTE_SCORES.PY EVALUATION
# ===============================================
print("="*60)
print("EVALUATING: compute_scores.py")
print("="*60)

# Block 1: Import statements
try:
    import torch
    from transformers import AutoTokenizer
    from transformer_lens import HookedTransformer
    import json
    from torch.nn import functional as F
    from typing import Dict, List, Tuple
    from sentence_transformers import SentenceTransformer
    import numpy as np
    import pandas as pd
    import argparse
    import sys
    import os
    import gc
    from tqdm import tqdm
    import matplotlib.pyplot as plt
    import seaborn as sns
    from scipy.stats import pointbiserialr
    
    record_eval("compute_scores.py", "B1", "Import statements", "Y", "Y", "N", "N")
except Exception as e:
    record_eval("compute_scores.py", "B1", "Import statements", "N", "N", "N", "N", str(e))

EVALUATING: compute_scores.py


/home/smallyan/.conda/envs/meta/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


[PASS] compute_scores.py:B1 - Import statements


In [4]:
# Block 2: load_examples function
try:
    def load_examples(file_path):
        """Load examples from JSONL file"""
        examples = []
        with open(file_path, 'r') as f:
            for line in f:
                data = json.loads(line)
                examples.append(data)
        return examples

    # Test with existing data
    test_path = "/net/scratch2/smallyan/InterpDetect_eval/scripts/preprocess/datasets/test/test1176_w_labels_filtered.jsonl"
    examples = load_examples(test_path)
    assert len(examples) > 0, "No examples loaded"
    assert 'prompt' in examples[0], "Missing 'prompt' field"
    record_eval("compute_scores.py", "B2", "load_examples function", "Y", "Y", "N", "N")
except Exception as e:
    record_eval("compute_scores.py", "B2", "load_examples function", "N", "N", "N", "N", str(e))

[PASS] compute_scores.py:B2 - load_examples function


In [5]:
# Block 3: calculate_dist_2d function (Jensen-Shannon divergence)
try:
    def calculate_dist_2d(sep_vocabulary_dist, sep_attention_dist):
        """Calculate Jensen-Shannon divergence between distributions"""
        softmax_mature_layer = F.softmax(sep_vocabulary_dist, dim=-1)
        softmax_anchor_layer = F.softmax(sep_attention_dist, dim=-1)
        M = 0.5 * (softmax_mature_layer + softmax_anchor_layer)
        log_softmax_mature_layer = F.log_softmax(sep_vocabulary_dist, dim=-1)
        log_softmax_anchor_layer = F.log_softmax(sep_attention_dist, dim=-1)
        kl1 = F.kl_div(log_softmax_mature_layer, M, reduction='none').sum(dim=-1)
        kl2 = F.kl_div(log_softmax_anchor_layer, M, reduction='none').sum(dim=-1)
        js_divs = 0.5 * (kl1 + kl2)
        scores = js_divs.cpu().tolist()
        return sum(scores)

    # Test the function
    test_dist1 = torch.randn(5, 100)
    test_dist2 = torch.randn(5, 100)
    result = calculate_dist_2d(test_dist1, test_dist2)
    assert isinstance(result, float), "Result should be a float"
    assert result >= 0, "JS divergence should be non-negative"
    record_eval("compute_scores.py", "B3", "calculate_dist_2d (JS divergence)", "Y", "Y", "N", "N")
except Exception as e:
    record_eval("compute_scores.py", "B3", "calculate_dist_2d (JS divergence)", "N", "N", "N", "N", str(e))

[PASS] compute_scores.py:B3 - calculate_dist_2d (JS divergence)


In [6]:
# Block 4: add_special_template function
try:
    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
    
    def add_special_template(tokenizer, prompt):
        """Add special template to prompt"""
        messages = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        return text

    # Test the function
    test_prompt = "What is the capital of France?"
    result = add_special_template(tokenizer, test_prompt)
    assert isinstance(result, str), "Result should be a string"
    assert "France" in result, "Prompt content should be preserved"
    record_eval("compute_scores.py", "B4", "add_special_template function", "Y", "Y", "N", "N")
except Exception as e:
    record_eval("compute_scores.py", "B4", "add_special_template function", "N", "N", "N", "N", str(e))

In [7]:
# Check last block status
print("Checking block 4 status...")